# 🚀 NameTag 백엔드 동작 원리 및 변수 흐름

이 노트북은 **NameTag 백엔드의 전체 작동 방식**을 단계별로 설명합니다.

## 📋 목차
1. **필수 라이브러리 임포트** - 백엔드 작동에 필요한 모든 모듈
2. **백엔드 모듈 구조** - 주요 구성 요소와 역할
3. **요청 처리 흐름** - HTTP 요청부터 응답까지
4. **변수 및 데이터 흐름** - 데이터 변환 과정
5. **응답 생성 과정** - 최종 응답 형식 생성
6. **에러 처리 메커니즘** - 예외 상황 대응

---

## 🏗️ 백엔드 시스템 아키텍처

```
┌─────────────────────────────────────────────────────────────────┐
│                     클라이언트 (Frontend)                          │
│                   (React + Vite + TypeScript)                   │
└────────────────────────┬────────────────────────────────────────┘
                         │ HTTP Request
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│                    FastAPI 백엔드 (main.py)                       │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │              Router 계층 (routers/brand.py)              │  │
│  │  ┌────────────────────────────────────────────────────┐  │  │
│  │  │ POST /api/v1/brand/generate                        │  │  │
│  │  │ POST /api/v1/brand/logo-variables                 │  │  │
│  │  │ GET /health                                        │  │  │
│  │  └────────────────────────────────────────────────────┘  │  │
│  └────────────────┬─────────────────────────────────────────┘  │
│                   │                                              │
│  ┌────────────────▼─────────────────────────────────────────┐  │
│  │         Service 계층 (services/brand_generator.py)       │  │
│  │  ┌────────────────────────────────────────────────────┐  │  │
│  │  │ • generate_brand_identity()                        │  │  │
│  │  │ • generate_logo_variables()                        │  │  │
│  │  │ • generate_logo_image()                            │  │  │
│  │  │ • generate_character_image()                       │  │  │
│  │  └────────────────────────────────────────────────────┘  │  │
│  └────────────────┬──────────────────────────────────────────┘  │
│                   │                                              │
│  ┌────────────────▼─────────────────────────────────────────┐  │
│  │         외부 API 호출 (Google Gemini API)                  │  │
│  │  • gemini-1.5-pro (텍스트 생성)                           │  │
│  │  • gemini-3.1-flash-image (이미지 생성)                   │  │
│  └────────────────┬──────────────────────────────────────────┘  │
│                   │                                              │
│  ┌────────────────▼─────────────────────────────────────────┐  │
│  │           Utility 계층 (utils/)                           │  │
│  │  • parser.py (JSON 추출)                                │  │
│  │  • logger.py (로깅 및 추적)                              │  │
│  │  • prompt_loader.py (프롬프트 템플릿)                   │  │
│  └──────────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
                         │ HTTP Response
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│                   클라이언트 (Frontend)                           │
│              (JSON 응답 수신 및 렌더링)                           │
└─────────────────────────────────────────────────────────────────┘
```

---

# 1️⃣ 필수 라이브러리 및 모듈 임포트

백엔드 작동에 필요한 모든 라이브러리와 모듈입니다.

## 📦 핵심 라이브러리

In [1]:
# 핵심 라이브러리
import json
import os
import uuid
from pathlib import Path
from typing import Any, Optional, Dict, List
from datetime import datetime

# 데이터 처리
import pandas as pd

# 웹 프레임워크
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# Google Gemini API
from google import genai
from google.genai import types

# PIL (이미지 처리)
from PIL import Image
import base64
import io

# 환경 설정
from dotenv import load_dotenv

print("✅ 모든 필수 라이브러리 로드 완료!")
print("\n📚 주요 라이브러리:")
print("  • FastAPI - REST API 프레임워크")
print("  • Pydantic - 데이터 검증 및 타입 체크")
print("  • Google Genai - AI 모델 API")
print("  • PIL - 이미지 처리")
print("  • Pandas - 데이터 분석")
print("  • Dotenv - 환경 변수 관리")

ModuleNotFoundError: No module named 'fastapi'

# 2️⃣ 백엔드 모듈 구조 분석

## 📁 디렉토리 구조와 역할

In [ ]:
# 백엔드 모듈 구조 정의
backend_modules = {
    "main.py": {
        "역할": "FastAPI 앱 초기화 및 라우터 연결",
        "주요_역할": [
            "FastAPI 인스턴스 생성",
            "CORS 미들웨어 설정",
            "라우터 등록 (router.include_router)",
            "에러 핸들러 설정",
            "헬스 체크 엔드포인트"
        ],
        "주요_코드": "app = FastAPI(...), app.include_router(brand.router, prefix='/api/v1')"
    },
    "routers/brand.py": {
        "역할": "HTTP 요청 처리 및 응답",
        "주요_역할": [
            "POST /api/v1/brand/generate - 브랜드 정체성 생성",
            "POST /api/v1/brand/logo-variables - 로고 변수 생성",
            "Pydantic 모델을 통한 요청/응답 검증",
            "HTTP 예외 처리"
        ],
        "주요_함수": [
            "generate_brand()",
            "get_logo_variables()",
            "health_check()"
        ]
    },
    "services/brand_generator.py": {
        "역할": "비즈니스 로직 및 AI API 호출",
        "주요_역할": [
            "Gemini API 호출",
            "JSON 응답 파싱",
            "AI 프롬프트 구성",
            "AI 응답 저장"
        ],
        "주요_함수": [
            "generate_brand_identity() - 브랜드명, 타이포그래피, 캐릭터 생성",
            "generate_logo_variables() - 로고 생성 변수 추천",
            "generate_logo_image() - 로고 이미지 생성",
            "generate_character_image() - 캐릭터 이미지 생성"
        ]
    },
    "utils/parser.py": {
        "역할": "AI 응답에서 JSON 추출",
        "주요_함수": ["extract_json() - 텍스트에서 JSON 파싱"]
    },
    "utils/logger.py": {
        "역할": "로깅 및 변수 추적",
        "주요_함수": [
            "setup_logger() - 로거 설정",
            "get_logger() - 로거 인스턴스",
            "get_tracker() - 세션별 변수 추적"
        ]
    },
    "utils/prompt_loader.py": {
        "역할": "프롬프트 템플릿 관리",
        "주요_함수": [
            "get_logo_prompt_template() - 로고 생성 프롬프트",
            "get_character_prompt_template() - 캐릭터 생성 프롬프트",
            "replace_variables() - 변수 치환"
        ]
    }
}

# 모듈 구조를 테이블로 표시
print("=" * 80)
print("📦 백엔드 모듈 구조")
print("=" * 80)

for module, info in backend_modules.items():
    print(f"\n✅ {module}")
    print(f"   역할: {info['역할']}")
    print(f"   주요 기능:")
    # print(f"   주요 역할:{info}")
    if '주요_역할' in info:
        for feature in info['주요_역할']:
            print(f"     • {feature}")
    
    if '주요_함수' in info:
        print(f"   주요 함수:")
        for func in info['주요_함수']:
            print(f"     • {func}")

📦 백엔드 모듈 구조

✅ main.py
   역할: FastAPI 앱 초기화 및 라우터 연결
   주요 기능:
     • FastAPI 인스턴스 생성
     • CORS 미들웨어 설정
     • 라우터 등록 (router.include_router)
     • 에러 핸들러 설정
     • 헬스 체크 엔드포인트

✅ routers/brand.py
   역할: HTTP 요청 처리 및 응답
   주요 기능:
     • POST /api/v1/brand/generate - 브랜드 정체성 생성
     • POST /api/v1/brand/logo-variables - 로고 변수 생성
     • Pydantic 모델을 통한 요청/응답 검증
     • HTTP 예외 처리
   주요 함수:
     • generate_brand()
     • get_logo_variables()
     • health_check()

✅ services/brand_generator.py
   역할: 비즈니스 로직 및 AI API 호출
   주요 기능:
     • Gemini API 호출
     • JSON 응답 파싱
     • AI 프롬프트 구성
     • AI 응답 저장
   주요 함수:
     • generate_brand_identity() - 브랜드명, 타이포그래피, 캐릭터 생성
     • generate_logo_variables() - 로고 생성 변수 추천
     • generate_logo_image() - 로고 이미지 생성
     • generate_character_image() - 캐릭터 이미지 생성

✅ utils/parser.py
   역할: AI 응답에서 JSON 추출
   주요 기능:
   주요 함수:
     • extract_json() - 텍스트에서 JSON 파싱

✅ utils/logger.py
   역할: 로깅 및 변수 추적
   주요 기능:
   주요 함수:
     • setup_logger() - 로거 설정
    

# 3️⃣ 요청 처리 흐름 (Request Flow)

## 🔄 단계별 요청 처리 과정

### 시나리오: POST /api/v1/brand/logo-variables 호출

In [ ]:
# 요청 처리 흐름 시뮬레이션
print("=" * 100)
print("🔄 HTTP 요청 처리 흐름")
print("=" * 100)

request_flow = [
    {
        "단계": 1,
        "위치": "Frontend (React)",
        "작업": "HTTP 요청 생성",
        "상세": "POST /api/v1/brand/logo-variables",
        "데이터": {
            "business_type": "온라인 쇼핑몰",
            "vibes": ["모던", "신뢰"],
            "target": "20-30대 직장인",
            "keywords": "지속가능한"
        }
    },
    {
        "단계": 2,
        "위치": "FastAPI (main.py)",
        "작업": "요청 라우팅",
        "상세": "CORS 미들웨어 확인 → 라우터 매칭",
        "검증": "요청 출처(localhost:3000) 허용 확인"
    },
    {
        "단계": 3,
        "위치": "Router (routers/brand.py)",
        "작업": "요청 데이터 검증",
        "상세": "Pydantic 모델 검증",
        "모델": "LogoVariablesRequest",
        "검증_항목": [
            "business_type: 최소 3글자",
            "vibes: 1-4개 항목",
            "target: 최소 3글자"
        ]
    },
    {
        "단계": 4,
        "위치": "Service (services/brand_generator.py)",
        "작업": "generate_logo_variables() 호출",
        "상세": "세션 ID 생성 → AI 프롬프트 구성",
        "변수": {
            "session_id": "uuid",
            "business_type": "온라인 쇼핑몰",
            "vibes": ["모던", "신뢰"],
            "target": "20-30대 직장인"
        }
    },
    {
        "단계": 5,
        "위치": "Gemini API",
        "작업": "AI 모델 호출",
        "상세": "ggemini-3-flash-preview 모델 사용",
        "입력": "프롬프트 + 시스템 인스트럭션",
        "출력": "JSON 형식의 AI 응답"
    },
    {
        "단계": 6,
        "위치": "Service (brand_generator.py)",
        "작업": "AI 응답 처리",
        "상세": "JSON 파싱 → 변수 추출 → 무드별 자동 매핑",
        "처리_단계": [
            "1. extract_json() - 텍스트에서 JSON 추출",
            "2. 필수 키 검증 (brand_identity, logo_variables)",
            "3. MOOD_MAPPING - 무드에 따른 기본값 설정",
            "4. 로그 저장 (JSON 파일)"
        ]
    },
    {
        "단계": 7,
        "위치": "Router (routers/brand.py)",
        "작업": "응답 생성",
        "상세": "Pydantic 모델로 응답 직렬화",
        "응답_모델": "LogoVariablesResponse"
    },
    {
        "단계": 8,
        "위치": "FastAPI",
        "작업": "HTTP 응답 반환",
        "상세": "JSON 직렬화 → HTTP 200 OK",
        "헤더": "Content-Type: application/json"
    },
    {
        "단계": 9,
        "위치": "Frontend (React)",
        "작업": "응답 수신 및 렌더링",
        "상세": "JSON 파싱 → UI 업데이트",
        "사용처": "로고 변수를 기반으로 이미지 생성 요청"
    }
]

# 각 단계 출력
for step in request_flow:
    print(f"\n{'─' * 100}")
    print(f"📍 단계 {step['단계']}: {step['위치']}")
    print(f"   작업: {step['작업']}")
    print(f"   상세: {step['상세']}")
    
    if '데이터' in step:
        print(f"   📦 입력 데이터:")
        for key, value in step['데이터'].items():
            print(f"      • {key}: {value}")
    
    if '검증' in step:
        print(f"   ✓ {step['검증']}")
    
    if '검증_항목' in step:
        print(f"   ✓ 검증 항목:")
        for item in step['검증_항목']:
            print(f"      • {item}")
    
    if '변수' in step:
        print(f"   📊 변수:")
        for key, value in step['변수'].items():
            print(f"      • {key}: {value}")
    
    if '입력' in step:
        print(f"   ➜ 입력: {step['입력']}")
        print(f"   ➜ 출력: {step['출력']}")
    
    if '처리_단계' in step:
        print(f"   ⚙️ 처리 단계:")
        for proc_step in step['처리_단계']:
            print(f"      {proc_step}")
    
    if '출력' in step and '입력' not in step:
        print(f"   ➜ 출력: {step['출력']}")

print(f"\n{'─' * 100}")
print("✅ 요청 처리 완료!")
print("=" * 100)

🔄 HTTP 요청 처리 흐름

────────────────────────────────────────────────────────────────────────────────────────────────────
📍 단계 1: Frontend (React)
   작업: HTTP 요청 생성
   상세: POST /api/v1/brand/logo-variables
   📦 입력 데이터:
      • business_type: 온라인 쇼핑몰
      • vibes: ['모던', '신뢰']
      • target: 20-30대 직장인
      • keywords: 지속가능한

────────────────────────────────────────────────────────────────────────────────────────────────────
📍 단계 2: FastAPI (main.py)
   작업: 요청 라우팅
   상세: CORS 미들웨어 확인 → 라우터 매칭
   ✓ 요청 출처(localhost:3000) 허용 확인

────────────────────────────────────────────────────────────────────────────────────────────────────
📍 단계 3: Router (routers/brand.py)
   작업: 요청 데이터 검증
   상세: Pydantic 모델 검증
   ✓ 검증 항목:
      • business_type: 최소 3글자
      • vibes: 1-4개 항목
      • target: 최소 3글자

────────────────────────────────────────────────────────────────────────────────────────────────────
📍 단계 4: Service (services/brand_generator.py)
   작업: generate_logo_variables() 호출
   상세: 세션 ID 생성 → AI 프롬프

# 4️⃣ 변수 및 데이터 흐름 분석

## 📊 변수가 어떻게 변환되고 전달되는지 추적

In [ ]:
# 실제 데이터 흐름 시뮬레이션
print("=" * 100)
print("🔀 변수 변환 및 데이터 흐름 추적")
print("=" * 100)

# 1️⃣ 클라이언트에서 전송한 원본 요청
print("\n[1️⃣] 클라이언트 → 서버: 원본 요청")
print("─" * 100)
original_request = {
    "business_type": "온라인 쇼핑몰",
    "vibes": ["모던", "신뢰"],
    "target": "20-30대 직장인",
    "keywords": "지속가능한"
}
print(f"요청 데이터 (JSON):")
print(json.dumps(original_request, ensure_ascii=False, indent=2))

# 2️⃣ Pydantic 모델 검증 후
print("\n[2️⃣] Router: Pydantic 모델 검증 및 변환")
print("─" * 100)
validated_request = {
    **original_request,
    "session_id": str(uuid.uuid4())[:8],  # 세션 ID 생성
    "_validated": True,
    "_timestamp": datetime.now().isoformat()
}
print(f"검증된 요청 데이터:")
print(json.dumps(validated_request, ensure_ascii=False, indent=2))

# 3️⃣ Service 레이어로 전달
print("\n[3️⃣] Service: generate_logo_variables() 함수 호출")
print("─" * 100)
service_input = {
    "business_type": validated_request["business_type"],
    "vibes": validated_request["vibes"],
    "target": validated_request["target"],
    "keywords": validated_request.get("keywords", ""),
    "session_id": validated_request["session_id"]
}
print(f"Service 입력 변수:")
print(json.dumps(service_input, ensure_ascii=False, indent=2))

# 4️⃣ 프롬프트 구성
print("\n[4️⃣] Gemini API: 프롬프트 구성")
print("─" * 100)
prompt_template = f"""사용자가 다음 정보를 입력했습니다:

업종/서비스: {service_input['business_type']}
브랜드 감성: {', '.join(service_input['vibes'])}
타겟 고객: {service_input['target']}
추가 키워드: {service_input['keywords'] or '없음'}

이 정보를 바탕으로 로고 생성에 필요한 다음 변수들을 추천해주세요.
[JSON 형식으로 응답]"""

print(f"구성된 프롬프트 (일부):")
print(prompt_template[:300] + "...")

# 5️⃣ AI 응답 시뮬레이션
print("\n[5️⃣] Gemini API: AI 응답 (시뮬레이션)")
print("─" * 100)
ai_response_example = {
    "brand_identity": {
        "brand_name": "지속가능마켓",
        "brand_name_meaning": "지속가능한 가치와 시장의 만남",
        "brand_topic": "친환경 제품 온라인 쇼핑몰",
        "core_value": "지속가능성, 투명성",
        "slogan": "작은 선택이 만드는 큰 변화"
    },
    "logo_variables": {
        "target_mood": "자연 & 지속가능",
        "brand_color": "#2D6A4F",
        "font_style": "유기적 산세리프",
        "symbol_type": "유기적",
        "logo_type": "심볼+텍스트 조합",
        "background": "흰색"
    },
    "recommendations_reason": "자연친화적인 감성을 표현하기 위해..."
}
print(f"AI 응답 (JSON):")
print(json.dumps(ai_response_example, ensure_ascii=False, indent=2))

# 6️⃣ 파싱 및 변수 추출
print("\n[6️⃣] Service: JSON 파싱 및 변수 추출")
print("─" * 100)
brand_identity = ai_response_example["brand_identity"]
logo_variables = ai_response_example["logo_variables"]

extracted_variables = {
    "BRAND_NAME": brand_identity["brand_name"],
    "BRAND_TOPIC": brand_identity["brand_topic"],
    "CORE_VALUE": brand_identity["core_value"],
    "LOGO_TARGET_MOOD": logo_variables["target_mood"],
    "LOGO_BRAND_COLOR": logo_variables["brand_color"],
    "LOGO_FONT_STYLE": logo_variables["font_style"],
    "LOGO_SYMBOL_TYPE": logo_variables["symbol_type"],
    "LOGO_TYPE": logo_variables["logo_type"],
    "LOGO_BACKGROUND": logo_variables["background"]
}

print(f"추출된 변수들:")
for var_name, var_value in extracted_variables.items():
    print(f"  {var_name}: {var_value}")

# 7️⃣ 무드 기반 자동 매핑
print("\n[7️⃣] Service: 무드 기반 자동 매핑 (MOOD_MAPPING)")
print("─" * 100)
mood_mapping = {
    "자연 & 지속가능": {
        "brand_color": "#2D6A4F",  # 녹색
        "font_style": "유기적 산세리프",
        "symbol_type": "유기적"
    }
}
selected_mood = logo_variables["target_mood"]
mood_defaults = mood_mapping.get(selected_mood, {})

print(f"선택된 무드: {selected_mood}")
print(f"자동 매핑된 기본값:")
print(json.dumps(mood_defaults, ensure_ascii=False, indent=2))

# 8️⃣ 최종 응답 구성
print("\n[8️⃣] Router: 최종 응답 구성")
print("─" * 100)
final_response = {
    "session_id": validated_request["session_id"],
    "brand_identity": brand_identity,
    "logo_variables": logo_variables,
    "recommendations_reason": ai_response_example["recommendations_reason"],
    "variables_summary": extracted_variables
}

print(f"최종 응답 (요약):")
print(f"  • session_id: {final_response['session_id']}")
print(f"  • brand_identity keys: {list(final_response['brand_identity'].keys())}")
print(f"  • logo_variables keys: {list(final_response['logo_variables'].keys())}")
print(f"  • tracked_variables_count: {len(final_response['variables_summary'])}")

print("\n" + "=" * 100)
print("✅ 데이터 변환 흐름 완료!")
print("=" * 100)

# 5️⃣ 두 가지 주요 엔드포인트 비교

## 🔀 generate_brand vs logo-variables의 차이

In [ ]:
# 두 엔드포인트 비교
import pandas as pd

endpoints_comparison = {
    "항목": [
        "URL",
        "HTTP 메서드",
        "목적",
        "AI 프롬프트",
        "응답 구조",
        "반환 데이터",
        "사용 시점",
        "데이터 복잡도",
        "처리 시간"
    ],
    "POST /brand/generate": [
        "POST /api/v1/brand/generate",
        "POST",
        "완전한 브랜드 정체성 생성",
        "build_user_prompt() 사용",
        "brands[], typography, character 포함",
        "3개 브랜드, 폰트, 캐릭터 정보",
        "초기 브랜드 구상 단계",
        "매우 높음",
        "약 10-15초"
    ],
    "POST /brand/logo-variables": [
        "POST /api/v1/brand/logo-variables",
        "POST",
        "로고 생성용 변수 추천",
        "build_logo_variables_prompt() 사용",
        "brand_identity, logo_variables 포함",
        "1개 브랜드, 로고 변수들, 무드 매핑",
        "로고/이미지 생성 직전",
        "낮음 (구조화된 변수)",
        "약 5-8초"
    ]
}

df_comparison = pd.DataFrame(endpoints_comparison)
print("=" * 130)
print("🔀 두 가지 엔드포인트 상세 비교")
print("=" * 130)
print(df_comparison.to_string(index=False))

# 상세 설명
print("\n\n" + "=" * 130)
print("📝 상세 설명")
print("=" * 130)

endpoint1_details = """
▶ POST /api/v1/brand/generate
─ 목적: 완전한 브랜드 정체성 생성
─ 입력: business_type, vibes[], target, keywords
─ 출력:
   1. brands[] - 3개의 완전한 브랜드 정보
      • name: 브랜드명
      • meaning: 이름의 의미
      • story: 브랜드 스토리
      • slogan: 슬로건
   2. typography - 권장 폰트
      • korean: 한글 폰트명
      • english: 영문 폰트명
      • reason: 선택 이유
   3. character - 마스코트 캐릭터
      • name: 캐릭터명
      • concept: 컨셉
      • personality: 성격
      • visual: 외형 묘사

─ 사용 흐름:
  1. 사용자 입력
  2. 3개의 브랜드 옵션 제시
  3. 사용자가 하나 선택
  4. 선택된 브랜드로 로고 생성
"""

endpoint2_details = """
▶ POST /api/v1/brand/logo-variables  
─ 목적: 로고 생성에 필요한 변수 추천
─ 입력: business_type, vibes[], target, keywords
─ 출력:
   1. brand_identity - 브랜드 정체성
      • brand_name: 추천 브랜드명
      • brand_topic: 사업 주제
      • core_value: 핵심 가치
      • slogan: 슬로건
   2. logo_variables - 로고 생성 변수
      • target_mood: 감성 (무드 코드)
      • brand_color: 색상 코드
      • font_style: 서체 스타일
      • symbol_type: 심볼 스타일
      • logo_type: 로고 구성
      • background: 배경 색상
   3. recommendations_reason - 추천 이유
   4. variables_summary - 모든 변수 요약

─ 사용 흐름:
  1. 사용자 입력
  2. AI가 최적의 로고 변수 추천
  3. MOOD_MAPPING으로 자동 매핑
  4. 로고 이미지 생성에 직접 사용
"""

print(endpoint1_details)
print(endpoint2_details)

print("\n" + "=" * 130)
print("💡 언제 어떤 엔드포인트를 써야 하는가?")
print("=" * 130)

usage_guide = """
┌─────────────────────────────────────────────────────────────────────────────────────┐
│ /brand/generate 사용:                                                               │
│  • 사용자에게 여러 브랜드 옵션을 제시해야 할 때                                      │
│  • 완전한 브랜드 정체성 (이름, 폰트, 캐릭터)이 필요할 때                             │
│  • 초기 브랜드 설계 단계                                                            │
└─────────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────────┐
│ /brand/logo-variables 사용:                                                         │
│  • 로고 이미지 생성 직전에 변수를 추천받을 때                                        │
│  • AI가 최적의 로고 생성 변수를 자동으로 구성해줄 때                                 │
│  • 로고/캐릭터 이미지 생성으로 바로 넘어갈 때                                        │
└─────────────────────────────────────────────────────────────────────────────────────┘
"""

print(usage_guide)

# 6️⃣ 응답 생성 과정

## 🔨 Pydantic 모델을 통한 응답 구성

In [ ]:
# Pydantic 모델을 이용한 응답 구성
print("=" * 100)
print("🔨 응답 생성 과정 (Pydantic 모델)")
print("=" * 100)

print("\n[1️⃣] 요청 검증 모델")
print("─" * 100)

request_models = {
    "LogoVariablesRequest": {
        "필드": [
            "business_type: str (최소 3글자) - 업종",
            "vibes: list[str] (1-4개) - 브랜드 감성",
            "target: str (최소 3글자) - 타겟 고객",
            "keywords: str (선택) - 추가 키워드",
            "session_id: str (선택) - 세션 ID"
        ],
        "역할": "클라이언트 요청 데이터 검증",
        "검증": "모든 필드가 지정된 규칙을 만족하는지 확인"
    },
    "BrandGenerateRequest": {
        "필드": [
            "business_type: str",
            "vibes: list[str]",
            "target: str",
            "keywords: str",
            "session_id: str (선택)"
        ],
        "역할": "브랜드 생성 요청 검증",
        "검증": "동일한 검증 로직"
    }
}

for model_name, model_info in request_models.items():
    print(f"\n✓ {model_name}")
    print(f"  역할: {model_info['역할']}")
    print(f"  필드:")
    for field in model_info['필드']:
        print(f"    • {field}")
    print(f"  검증: {model_info['검증']}")

print("\n[2️⃣] 응답 모델 - LogoVariablesResponse")
print("─" * 100)

response_structure = {
    "session_id": "str (UUID) - 요청/응답 추적 ID",
    "brand_identity": {
        "brand_name": "str - 추천 브랜드명",
        "brand_name_meaning": "str - 브랜드명의 의미",
        "brand_topic": "str - 구체적인 사업 주제",
        "core_value": "str - 핵심 가치",
        "slogan": "str - 슬로건"
    },
    "logo_variables": {
        "target_mood": "str - 브랜드 무드 (MOOD_MAPPING 키)",
        "brand_color": "str - 컬러 코드 (예: #2D6A4F)",
        "font_style": "str - 서체 스타일",
        "symbol_type": "str - 심볼 스타일",
        "logo_type": "str - 로고 구성 방식",
        "background": "str - 배경 색상"
    },
    "recommendations_reason": "str - 이 변수들을 추천한 이유",
    "variables_summary": "dict - 모든 변수 요약"
}

print("\n응답 구조:")
def print_response_structure(obj, indent=1):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, dict):
                print(f"{'  ' * indent}├─ {key}:")
                print_response_structure(value, indent + 1)
            else:
                print(f"{'  ' * indent}├─ {key}: {value}")
    elif isinstance(obj, list):
        for item in obj:
            print(f"{'  ' * indent}├─ {item}")

print_response_structure(response_structure)

print("\n[3️⃣] 응답 직렬화 과정")
print("─" * 100)

serialization_steps = [
    {
        "단계": 1,
        "위치": "Service 레이어",
        "작업": "Python dict 반환",
        "예시": "{'session_id': '...', 'brand_identity': {...}, ...}"
    },
    {
        "단계": 2,
        "위치": "Router 레이어",
        "작업": "response_model 매개변수 검증",
        "예시": "response_model=LogoVariablesResponse"
    },
    {
        "단계": 3,
        "위치": "Pydantic",
        "작업": "dict를 Pydantic 모델로 변환",
        "예시": "LogoVariablesResponse(**data)"
    },
    {
        "단계": 4,
        "위치": "FastAPI",
        "작업": "Pydantic 모델을 JSON으로 직렬화",
        "예시": "model.model_dump_json()"
    },
    {
        "단계": 5,
        "위치": "HTTP 응답",
        "작업": "JSON 반환",
        "예시": "Content-Type: application/json"
    },
    {
        "단계": 6,
        "위치": "클라이언트",
        "작업": "JSON 파싱 및 렌더링",
        "예시": "JSON.parse(response)"
    }
]

for step in serialization_steps:
    print(f"\n{step['단계']}. {step['위치']} - {step['작업']}")
    print(f"   예시: {step['예시']}")

print("\n[4️⃣] 실제 응답 예시 (브랜드별 3개 옵션)")
print("─" * 100)

example_response = {
    "session_id": "a1b2c3d4",
    "brand_identity": {
        "brand_name": "지속가능마켓",
        "brand_name_meaning": "지속가능한 가치와 시장의 만남을 의미합니다",
        "brand_topic": "친환경 제품 온라인 쇼핑몰",
        "core_value": "지속가능성, 투명성",
        "slogan": "작은 선택이 만드는 큰 변화"
    },
    "logo_variables": {
        "target_mood": "자연 & 지속가능",
        "brand_color": "#2D6A4F",
        "font_style": "유기적 산세리프",
        "symbol_type": "유기적",
        "logo_type": "심볼+텍스트 조합",
        "background": "흰색"
    },
    "recommendations_reason": "지속가능한 브랜드 이미지를 위해 자연친화적 감성의 녹색 계열을 추천했습니다",
    "variables_summary": {
        "BRAND_NAME": "지속가능마켓",
        "BRAND_TOPIC": "친환경 제품 온라인 쇼핑몰",
        "CORE_VALUE": "지속가능성, 투명성",
        "LOGO_TARGET_MOOD": "자연 & 지속가능",
        "LOGO_BRAND_COLOR": "#2D6A4F",
        "LOGO_FONT_STYLE": "유기적 산세리프",
        "LOGO_SYMBOL_TYPE": "유기적",
        "LOGO_TYPE": "심볼+텍스트 조합",
        "LOGO_BACKGROUND": "흰색"
    }
}

print("JSON 응답 (예시):")
print(json.dumps(example_response, ensure_ascii=False, indent=2))

print("\n" + "=" * 100)

# 7️⃣ 에러 처리 메커니즘

## 🚨 예외 상황 처리 및 에러 응답

In [ ]:
# 에러 처리 메커니즘 분석
print("=" * 100)
print("🚨 에러 처리 메커니즘")
print("=" * 100)

# 에러 타입별 처리 방식
error_handling_map = {
    "요청 검증 에러": {
        "원인": [
            "business_type < 3글자",
            "vibes 배열 비어있음",
            "target < 3글자"
        ],
        "처리": "Pydantic 자동 검증",
        "HTTP_코드": 422,
        "응답": {
            "detail": [
                {
                    "loc": ["body", "business_type"],
                    "msg": "ensure this value has at least 3 characters",
                    "type": "value_error.any_str.min_length"
                }
            ]
        }
    },
    "API 키 누락": {
        "원인": [
            "GEMINI_API_KEY 환경 변수 없음",
            "GEMINI_IMAGE_API_KEY 환경 변수 없음"
        ],
        "처리": "RuntimeError 발생",
        "HTTP_코드": 500,
        "응답": {
            "detail": "GEMINI_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요."
        }
    },
    "JSON 파싱 실패": {
        "원인": [
            "AI 응답이 유효한 JSON이 아님",
            "필수 키가 응답에 포함되지 않음"
        ],
        "처리": "ValueError 발생",
        "HTTP_코드": 422,
        "응답": {
            "detail": "JSON 파싱 실패. 원본 응답: [...]"
        }
    },
    "Gemini API 에러": {
        "원인": [
            "API 호출 실패",
            "API 속도 제한 (Rate limit)",
            "API 응답 타임아웃"
        ],
        "처리": "Exception 캐치",
        "HTTP_코드": 500,
        "응답": {
            "detail": "로고 변수 생성 실패: [에러 메시지]"
        }
    }
}

print("\n[1️⃣] 에러 타입별 처리 방식\n")

for error_type, details in error_handling_map.items():
    print(f"📍 {error_type}")
    print(f"   HTTP 상태 코드: {details['HTTP_코드']}")
    print(f"   원인:")
    for cause in details['원인']:
        print(f"     • {cause}")
    print(f"   처리 방식: {details['처리']}")
    print()

print("\n[2️⃣] 에러 처리 플로우")
print("─" * 100)

error_flow = """
┌─────────────────────────────────────────────────────────────┐
│                    HTTP 요청 수신                            │
└────────────────────┬────────────────────────────────────────┘
                     │
                     ▼
          ┌──────────────────────┐
          │  CORS 검증           │
          │  (요청 출처 확인)      │
          └──────┬───────────────┘
                 │
         ┌───────▼──────────┐
         │ CORS 실패?        │
         └─┬────────────┬──┘
           │ 예          │ 아니오
           │            └────┐
           │                 ▼
           │         ┌───────────────────────┐
           │         │ Pydantic 검증        │
           │         │ (요청 데이터 검증)    │
           │         └──────┬────────────────┘
           │                │
           │        ┌───────▼──────────┐
           │        │ 검증 실패?       │
           │        └─┬────────┬──────┘
           │          │ 예     │ 아니오
           │          │        └────┐
           │          │             ▼
           │          │      ┌──────────────────┐
           │          │      │ Service 레이어  │
           │          │      │ (비즈니스 로직)  │
           │          │      └────┬─────────────┘
           │          │           │
           │          │    ┌──────▼──────────┐
           │          │    │ 실행 성공?     │
           │          │    └─┬────────┬─────┘
           │          │      │ 예     │ 아니오
           │          │      │        └────┐
           │          │      │             │
           │          │      │      ┌──────▼──────────┐
           │          │      │      │ Exception 캐치 │
           │          │      │      └────┬───────────┘
           │          │      │           │
           │          │      │     ┌─────▼────────┐
           │          │      │     │ 에러 로깅    │
           │          │      │     └────┬─────────┘
           │          │      │          │
    ┌──────▼──────────▼──────▼──────────▼──────────┐
    │         HTTP 에러 응답 반환                   │
    │  (422, 500 등 + JSON 에러 메시지)            │
    └──────────────────┬─────────────────────────┘
                       │
                       ▼
            ┌──────────────────────┐
            │  HTTP 성공 응답 반환  │
            │  (200 + JSON 데이터) │
            └──────────────────────┘
"""

print(error_flow)

print("\n[3️⃣] 백엔드 에러 처리 코드 패턴")
print("─" * 100)

error_handling_code = """
# routers/brand.py의 에러 처리 패턴

@router.post("/brand/logo-variables", response_model=LogoVariablesResponse)
async def get_logo_variables(request: LogoVariablesRequest) -> dict[str, Any]:
    try:
        session_id = request.session_id or str(uuid.uuid4())
        
        # Service 호출
        result = generate_logo_variables(
            business_type=request.business_type,
            vibes=request.vibes,
            target=request.target,
            keywords=request.keywords,
            session_id=session_id,
        )
        return result
        
    except ValueError as e:
        # JSON 파싱 실패, 필수 키 누락 등
        raise HTTPException(status_code=422, detail=str(e))
        
    except RuntimeError as e:
        # API 키 누락 등 설정 에러
        raise HTTPException(status_code=500, detail=str(e))
        
    except Exception as e:
        # 예상치 못한 에러
        raise HTTPException(status_code=500, detail=f"로고 변수 생성 실패: {str(e)}")
"""

print(error_handling_code)

print("\n[4️⃣] 에러 응답 예시")
print("─" * 100)

error_examples = {
    "❌ 요청 검증 실패 (422)": {
        "status_code": 422,
        "response": {
            "detail": [
                {
                    "loc": ["body", "business_type"],
                    "msg": "ensure this value has at least 3 characters",
                    "type": "value_error.any_str.min_length"
                }
            ]
        }
    },
    "❌ API 키 누락 (500)": {
        "status_code": 500,
        "response": {
            "detail": "GEMINI_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요."
        }
    },
    "❌ JSON 파싱 실패 (422)": {
        "status_code": 422,
        "response": {
            "detail": "JSON 파싱 실패. 원본 응답: {...}"
        }
    }
}

for error_name, error_info in error_examples.items():
    print(f"\n{error_name}")
    print(f"응답:")
    print(json.dumps(error_info["response"], ensure_ascii=False, indent=2))

print("\n" + "=" * 100)
print("✅ 에러 처리 메커니즘 이해 완료!")
print("=" * 100)

# 🎯 종합 요약 및 핵심 포인트

## 📚 백엔드 동작의 핵심 이해

In [ ]:
# 종합 요약
print("=" * 120)
print("🎯 NameTag 백엔드 동작 원리 - 종합 요약")
print("=" * 120)

summary = """

▶ 핵심 1️⃣: 요청-응답 사이클
   1. 클라이언트 → FastAPI 라우터
   2. 라우터 → Pydantic 검증
   3. 라우터 → Service 레이어
   4. Service → Gemini API 호출
   5. Gemini API → JSON 응답
   6. Service → 응답 파싱 및 변수 추출
   7. 라우터 → Pydantic 응답 모델 직렬화
   8. FastAPI → JSON HTTP 응답
   9. 클라이언트 → JSON 파싱 및 렌더링

▶ 핵심 2️⃣: 변수의 흐름
   요청 데이터 (4개 변수)
       ↓
   Pydantic 검증 & 세션 ID 추가
       ↓
   프롬프트 구성 (변수 → 프롬프트 텍스트)
       ↓
   Gemini API 호출
       ↓
   JSON 응답 수신
       ↓
   extract_json() - 텍스트에서 JSON 추출
       ↓
   변수 추출 및 매핑
       ↓
   MOOD_MAPPING - 무드에 따른 자동값 설정
       ↓
   최종 응답 생성
       ↓
   클라이언트로 전송

▶ 핵심 3️⃣: 두 가지 주요 엔드포인트
   
   1. POST /api/v1/brand/generate
      목적: 완전한 브랜드 정체성 (이름, 폰트, 캐릭터)
      반환: 3개의 완전한 브랜드 옵션
      사용시점: 초기 브랜드 설계 단계
      
   2. POST /api/v1/brand/logo-variables
      목적: 로고 생성용 변수 추천
      반환: 1개의 최적화된 로고 변수 세트
      사용시점: 로고/이미지 생성 직전

▶ 핵심 4️⃣: 데이터 검증 레이어
   • Pydantic: 요청 데이터 스키마 검증
   • 필수 필드 체크: business_type, vibes, target
   • 데이터 타입 체크: str, list[str] 등
   • 길이 검증: 최소 글자수 확인
   • 자동 에러 응답 (HTTP 422)

▶ 핵심 5️⃣: 에러 처리 계층
   • ValueError: JSON 파싱 실패 → HTTP 422
   • RuntimeError: API 키 누락 → HTTP 500
   • Exception: 예상치 못한 에러 → HTTP 500
   • 모든 에러는 로깅되고 명확한 메시지와 함께 반환

▶ 핵심 6️⃣: 세션 추적 (Session Tracking)
   • 모든 요청에 session_id 부여 (UUID)
   • 요청-응답을 세션 ID로 연결
   • 로그에 저장되어 요청 추적 가능
   • 변수들이 변수_트래커에 기록됨

▶ 핵심 7️⃣: MOOD_MAPPING (감성별 자동 매핑)
   
   무드 → 기본값 자동 설정
   ├─ "자연 & 지속가능" → #2D6A4F (녹색), "유기적"
   ├─ "모던 & 신뢰" → #333333 (검정), "기하학적"
   ├─ "따뜻 & 친근" → #C8553D (따뜻한 갈색), "유기적"
   └─ ... (총 8가지 무드)
   
   사용자가 지정하지 않은 값은 무드에 따라 자동 설정됨

▶ 핵심 8️⃣: 로깅 시스템
   • 모든 단계에서 로그 기록
   • AI 응답 전체 저장 (txt + JSON 파일)
   • 변수 추적 시스템으로 세션별 변수 관리
   • 디버깅 및 문제 분석에 활용

"""

print(summary)

print("\n" + "=" * 120)
print("📋 주요 함수 및 역할 정리")
print("=" * 120)

functions_overview = {
    "main.py": {
        "역할": "FastAPI 앱 초기화",
        "주요 작업": [
            "FastAPI 인스턴스 생성",
            "CORS 설정",
            "라우터 등록",
            "헬스 체크 제공"
        ]
    },
    "routers/brand.py": {
        "역할": "HTTP 요청 처리",
        "주요_함수": [
            "generate_brand() - 브랜드 생성 요청 처리",
            "get_logo_variables() - 로고 변수 요청 처리"
        ]
    },
    "services/brand_generator.py": {
        "역할": "비즈니스 로직 구현",
        "주요_함수": [
            "generate_logo_variables()",
            "generate_brand_identity()",
            "generate_logo_image()",
            "generate_character_image()"
        ]
    },
    "utils/parser.py": {
        "역할": "JSON 파싱",
        "주요_함수": ["extract_json()"]
    },
    "utils/logger.py": {
        "역할": "로깅 및 변수 추적",
        "주요_함수": [
            "get_logger()",
            "get_tracker()",
            "setup_logger()"
        ]
    }
}

for file_name, info in functions_overview.items():
    print(f"\n✓ {file_name}")
    print(f"  역할: {info['역할']}")
    if '주요_함수' in info:
        print(f"  함수:")
        for func in info['주요_함수']:
            print(f"    • {func}")
    if '주요 작업' in info:
        print(f"  작업:")
        for task in info['주요 작업']:
            print(f"    • {task}")

print("\n" + "=" * 120)
print("✨ 이제 백엔드의 전체 흐름을 이해했습니다!")
print("=" * 120)

# 🧪 실습: 전체 흐름 시뮬레이션

## 실제 데이터로 백엔드 작동 추적하기

In [3]:
"""
실제 사용자 입력 + AI 응답 + 이미지 생성
이 셀에서는 시뮬레이션이 아닌 실제 작동하는 코드입니다.
"""

import sys
import os
from pathlib import Path

# Add backend directory to Python path
backend_path = Path(".").resolve().parent / "backend"
sys.path.insert(0, str(backend_path))
sys.path.insert(0, str(Path(".").resolve().parent))

# Import necessary modules
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
import base64
import io
import json
from datetime import datetime
import uuid

# Load environment variables
load_dotenv()

# Initialize Gemini client
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_IMAGE_API_KEY = os.getenv("GEMINI_IMAGE_API_KEY")

if not GEMINI_API_KEY:
    print("❌ GEMINI_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요.")
    sys.exit()
else:
    print("✅ Gemini API 키 로드 완료")

print("\n" + "=" * 100)
print("🎯 NameTag 브랜드 생성 (실제 AI + 이미지 생성)")
print("=" * 100)

# 사용자 입력 받기
print("\n📝 브랜드 정보를 입력해주세요:\n")

business_type = "20대를 위한 감성 소품 온라인 셀렉샵"  # Simulated input
# business_type = input("🏢 업종/서비스를 입력하세요 (예: 온라인 친환경 쇼핑몰): ").strip()
while len(business_type) < 3:
    business_type = input("❌ 최소 3글자 이상 입력해주세요: ").strip()

vibes_input = "따뜻한, 감성적인, 자연친화적, 레트로한"  # Simulated input
# vibes_input = input("✨ 브랜드 감성을 선택하세요 (쉼표로 구분, 예: 모던, 신뢰): ").strip()
vibes = [v.strip() for v in vibes_input.split(",") if v.strip()]
while len(vibes) == 0 or len(vibes) > 4:
    vibes_input = input("❌ 1~4개의 감성을 선택해주세요: ").strip()
    vibes = [v.strip() for v in vibes_input.split(",") if v.strip()]

target = "30대 직장 여성, 소소한 취미생활을 즐기고 자신만의 공간을 꾸미는 것에 관심 있는 분들"  # Simulated input
# target = input("👥 타겟 고객을 입력하세요 (예: 20-40대 환경 의식 있는 소비자): ").strip()
while len(target) < 3:
    target = input("❌ 최소 3글자 이상 입력해주세요: ").strip()

keywords = "따뜻함, 일상, 발견"  # Simulated input
# keywords = input("🔑 추가 키워드 (선택사항): ").strip()

print("\n" + "=" * 100)
print("✅ 입력된 브랜드 정보:")
print(f"업종/서비스: {business_type}")
print(f"브랜드 감성: {', '.join(vibes)}")
print(f"타겟 고객: {target}")
print(f"추가 키워드: {keywords}")

✅ Gemini API 키 로드 완료

🎯 NameTag 브랜드 생성 (실제 AI + 이미지 생성)

📝 브랜드 정보를 입력해주세요:


✅ 입력된 브랜드 정보:
업종/서비스: 20대를 위한 감성 소품 온라인 셀렉샵
브랜드 감성: 따뜻한, 감성적인, 자연친화적, 레트로한
타겟 고객: 30대 직장 여성, 소소한 취미생활을 즐기고 자신만의 공간을 꾸미는 것에 관심 있는 분들
추가 키워드: 따뜻함, 일상, 발견


In [4]:
print("\n" + "-" * 100)
print("💬 AI에 요청 중입니다... 잠시만 기다려주세요.")
print("-" * 100)

# 1️⃣ Gemini API에 요청
client = genai.Client(api_key=GEMINI_API_KEY)

prompt = f"""당신은 전문 브랜드 디렉터입니다.
사용자의 정보를 바탕으로 초기 스타트업/1인 기업의 브랜드 정체성을 설계합니다.
반드시 지정된 JSON 형식만 출력하고, 다른 텍스트는 포함하지 마세요.
브랜드의 통실성이 중요하므로, 입력된 정보를 최대한 활용하여 브랜드 아이덴티티를 구성하세요.
각 브랜드는 고객 감성을 자극하는 스토리를 포함해야 합니다.
각 브랜드는 통일성 있는 브랜드명, 캐릭터, 폰트, 로고 스타일을 가져야 합니다.

업종/서비스: {business_type}
브랜드 감성: {', '.join(vibes)}
타겟 고객: {target}
추가 키워드: {keywords or '없음'}

아래 JSON 형식으로만 응답하세요:

{{
  "brands": [
    {{
      "name": "브랜드명",
      "meaning": "이름의 의미와 어원 1-2문장",
      "story": "고객 감성을 자극하는 스토리 2-3문장",
      "slogan": "핵심 슬로건 10단어 이내"
    }},
    {{ "name": "두번째 브랜드명", "meaning": "...", "story": "...", "slogan": "..." }},
    {{ "name": "세번째 브랜드명", "meaning": "...", "story": "...", "slogan": "..." }}
  ],
  "typography": [
    {{
      "korean": "추천 한글 폰트명",
      "english": "추천 영문 폰트명",
      "reason": "선택 이유 2문장"
    }},
    {{
      "korean": "두번째 한글 폰트명",
      "english": "두번째 영문 폰트명",
      "reason": "선택 이유 2문장"
    }},
    {{
      "korean": "세번째 한글 폰트명",
      "english": "세번째 영문 폰트명",
      "reason": "선택 이유 2문장"
    }}
  ],
  "character": [
    {{
      "name": "캐릭터 이름",
      "concept": "컨셉 한 줄",
      "personality": "성격 특징",
      "visual": "외형 묘사 2-3문장"
    }},
    {{
      "name": "두번째 캐릭터 이름",
      "concept": "컨셉 한 줄",
      "personality": "성격 특징",
      "visual": "외형 묘사 2-3문장"
    }},
    {{
      "name": "세번째 캐릭터 이름",
      "concept": "컨셉 한 줄",
      "personality": "성격 특징",
      "visual": "외형 묘사 2-3문장"
    }}
  ],
  "logo_design": [
    {{
      "style": "로고 스타일 (예: 심플하고 모던한)",
      "brand_color": "로고 색상 (예: #2D6A4F)",
      "symbol_type": "심볼 스타일",
      "logo_type": "로고 구성 (예: 심볼+텍스트 조합)",
      "font_style": "서체 스타일 (예: 유기적 산세리프)",
      "font_weight": "폰트 두께 (예: 보통, 굵음)",
      "background": "배경색 (예: 흰색)"
    }},
    {{
      "style": "두번째 로고 스타일",
      "brand_color": "두번째 로고 색상",
      "symbol_type": "심볼 스타일",
      "logo_type": "로고 구성",
      "font_style": "서체 스타일",
      "font_weight": "폰트 두께",
      "background": "배경색"
    }},
    {{
      "style": "세번째 로고 스타일",
      "brand_color": "세번째 로고 색상",
      "symbol_type": "심볼 스타일",
      "logo_type": "로고 구성",
      "font_style": "서체 스타일",
      "font_weight": "폰트 두께",
      "background": "배경색"
    }}
  ]
}}"""


SYSTEM_PROMPT = """당신은 전문 브랜드 디렉터입니다.
사용자의 정보를 바탕으로 초기 스타트업/1인 기업의 브랜드 정체성을 설계합니다.
반드시 지정된 JSON 형식만 출력하고, 다른 텍스트는 포함하지 마세요."""

try:
    main_infomation_response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT),
    )
    main_infomation_raw_text = main_infomation_response.text or ""
    print("\n✅ AI 응답 수신 완료!")
    print(f"응답 텍스트: {main_infomation_raw_text}...")
    
except Exception as e:
    print(f"\n❌ AI 응답 오류: {e}")
    main_infomation_raw_text = ""
    sys.exit()


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "brands": [
    {
      "name": "온온(OnOn)",
      "meaning": "따뜻할 온(溫)을 겹쳐 사용하여, 공간에 온기를 겹겹이 쌓아준다는 의미를 담고 있습니다.",
      "story": "퇴근 후 불 꺼진 방을 채우는 작은 스탠드 불빛처럼, 지친 당신의 일상을 안아주는 사소하지만 소중한 소품들을 제안합니다. 나만의 온도로 채워진 공간에서 비로소 완성되는 진정한 휴식을 발견해보세요.",
      "slogan": "당신의 공간에 온기를 채우는 시간"
    },
    {
      "name": "늘봄 아뜰리에",
      "meaning": "언제나 봄과 같은 따스함과 생명력을 지향하는 감성 편집숍이라는 뜻입니다.",
      "story": "차가운 도심 속에서도 계절의 변화를 느끼고 싶은 당신을 위해 자연의 질감과 색채를 닮은 소품들을 큐레이션합니다. 바쁜 업무 끝에 마주하는 작은 화분과 빈티지한 소품들이 당신의 일상을 매일 봄으로 만들어줍니다.",
      "slogan": "일상이 머무는 자리, 사계절의 따스함"
    },
    {
      "name": "오디(Odi)",
      "meaning": "'Ordinary Discovery'의 약자로, 평범한 일상 속에서 나만의 취향을 발견한다는 의미입니다.",
      "story": "매일 똑같은 출퇴근길 사이, 우연히 발견한 보물 같은 소품 하나가 주는 설렘을 믿습니다. 레트로한 

In [5]:
# 2️⃣ JSON 파싱
print("\n📊 AI 응답 분석 중...")

try:
    # JSON 추출
    import re
    json_match = re.search(r'\{[\s\S]*\}', main_infomation_raw_text)
    if json_match:
        main_infomation_ai_response = json.loads(json_match.group())
    else:
        main_infomation_ai_response = json.loads(main_infomation_raw_text)

    print("✅ JSON 파싱 완료!")
    
except Exception as e:
    print(f"❌ JSON 파싱 오류: {e}")
    print(f"원본 응답: {main_infomation_raw_text}")
    main_infomation_ai_response = {}
    sys.exit()



📊 AI 응답 분석 중...
✅ JSON 파싱 완료!


In [6]:
# 3️⃣ 결과 표시
print("\n" + "=" * 100)
print("🎨 AI가 제안한 3가지 브랜드 옵션")
print("=" * 100)

if "brands" in main_infomation_ai_response and "typography" in main_infomation_ai_response and "character" in main_infomation_ai_response and "logo_design" in main_infomation_ai_response:
    brands = main_infomation_ai_response["brands"]
    typography = main_infomation_ai_response["typography"]
    character = main_infomation_ai_response["character"]
    logo_design = main_infomation_ai_response["logo_design"]
    for i, (brand, typography, character, logo_design) in enumerate(zip(brands, typography, character, logo_design), 1):
        print(f"\n[옵션 {i}] {brand.get('name', '브랜드명 없음')}")
        print(f"  📖 의미: {brand.get('meaning', '-')}")
        print(f"  📚 스토리: {brand.get('story', '-')}")
        print(f"  💬 슬로건: {brand.get('slogan', '-')}")
        print(f"  🎨 추천 폰트: {typography.get('font', '-')}")
        print(f"  🎭 캐릭터: {character.get('name', '-')}")
        print(f"  🎨 로고 디자인: {logo_design.get('style', '-')}")
        print(f"  🎨 로고 색상: {logo_design.get('brand_color', '-')}")
        print(f"  🎨 심볼 스타일: {logo_design.get('symbol_type', '-')}")
        print(f"  🎨 로고 구성: {logo_design.get('logo_type', '-')}")
        print(f"  🎨 배경색: {logo_design.get('background', '-')}")


🎨 AI가 제안한 3가지 브랜드 옵션

[옵션 1] 온온(OnOn)
  📖 의미: 따뜻할 온(溫)을 겹쳐 사용하여, 공간에 온기를 겹겹이 쌓아준다는 의미를 담고 있습니다.
  📚 스토리: 퇴근 후 불 꺼진 방을 채우는 작은 스탠드 불빛처럼, 지친 당신의 일상을 안아주는 사소하지만 소중한 소품들을 제안합니다. 나만의 온도로 채워진 공간에서 비로소 완성되는 진정한 휴식을 발견해보세요.
  💬 슬로건: 당신의 공간에 온기를 채우는 시간
  🎨 추천 폰트: -
  🎭 캐릭터: 포근이
  🎨 로고 디자인: 부드럽고 서정적인 엠블럼 스타일
  🎨 로고 색상: #D4A373
  🎨 심볼 스타일: 심플한 캔들 또는 램프 실루엣
  🎨 로고 구성: 심볼+텍스트 가로 조합
  🎨 배경색: #FAFAF5

[옵션 2] 늘봄 아뜰리에
  📖 의미: 언제나 봄과 같은 따스함과 생명력을 지향하는 감성 편집숍이라는 뜻입니다.
  📚 스토리: 차가운 도심 속에서도 계절의 변화를 느끼고 싶은 당신을 위해 자연의 질감과 색채를 닮은 소품들을 큐레이션합니다. 바쁜 업무 끝에 마주하는 작은 화분과 빈티지한 소품들이 당신의 일상을 매일 봄으로 만들어줍니다.
  💬 슬로건: 일상이 머무는 자리, 사계절의 따스함
  🎨 추천 폰트: -
  🎭 캐릭터: 리프(Leaf)
  🎨 로고 디자인: 빈티지한 스탬프 스타일
  🎨 로고 색상: #6B705C
  🎨 심볼 스타일: 나뭇잎과 꽃 모티프의 원형 테두리
  🎨 로고 구성: 원형 엠블럼 형태
  🎨 배경색: #F3EFE0

[옵션 3] 오디(Odi)
  📖 의미: 'Ordinary Discovery'의 약자로, 평범한 일상 속에서 나만의 취향을 발견한다는 의미입니다.
  📚 스토리: 매일 똑같은 출퇴근길 사이, 우연히 발견한 보물 같은 소품 하나가 주는 설렘을 믿습니다. 레트로한 감성과 현대적인 감각이 공존하는 이곳에서 당신의 숨겨진 취향을 조심스레 꺼내보세요.
  💬 슬로건: 평범한 하루를 빛내는 사소한 발견
  🎨 추천 폰트: -
  🎭 캐릭터: 디디

In [8]:
# 사용자가 브랜드 선택
print("\n" + "-" * 100)
selected_index = input("\n🎯 어느 브랜드를 선택하시겠습니까? (1, 2, 또는 3): ").strip()

try:
    selected_index = int(selected_index) - 1
    if 0 <= selected_index < len(brands):
        selected_brand = brands[selected_index]
        print(f"✅ '{selected_brand['name']}'이(가) 선택되었습니다!")
    else:
        selected_brand = brands[0]
        print(f"⚠️ 잘못된 입력입니다. '{brands[0]['name']}'을(를) 선택하겠습니다.")
except:
    selected_brand = brands[0]
    print(f"⚠️ 입력 오류. '{brands[0]['name']}'을(를) 선택하겠습니다.")

# 4️⃣ 타이포그래피 정보
print("\n" + "=" * 100)
print("🔤 추천 타이포그래피")
print("=" * 100)

if "typography" in main_infomation_ai_response and isinstance(main_infomation_ai_response["typography"], list) and len(main_infomation_ai_response["typography"]) > 0:
    typography = main_infomation_ai_response["typography"][0]
    print(f"\n한글 폰트: {typography.get('korean', '-')}")
    print(f"영문 폰트: {typography.get('english', '-')}")
    print(f"선택 이유: {typography.get('reason', '-')}")

# 5️⃣ 캐릭터 정보
print("\n" + "=" * 100)
print("🎭 마스코트 캐릭터")
print("=" * 100)

if "character" in main_infomation_ai_response and isinstance(main_infomation_ai_response["character"], list) and len(main_infomation_ai_response["character"]) > 0:
    character = main_infomation_ai_response["character"][0]
    print(f"\n캐릭터명: {character.get('name', '-')}")
    print(f"컨셉: {character.get('concept', '-')}")
    print(f"성격: {character.get('personality', '-')}")
    print(f"외형: {character.get('visual', '-')}")

# 6️⃣ 로고 디자인 정보
print("\n" + "=" * 100)
print("🎨 로고 디자인")
print("=" * 100)

if "logo_design" in main_infomation_ai_response and isinstance(main_infomation_ai_response["logo_design"], list) and len(main_infomation_ai_response["logo_design"]) > 0:
    logo_design = main_infomation_ai_response["logo_design"][0]
    print(f"\n로고 스타일: {logo_design.get('style', '-')}")
    print(f"로고 색상: {logo_design.get('brand_color', '-')}")
    print(f"심볼 스타일: {logo_design.get('symbol_type', '-')}")
    print(f"로고 구성: {logo_design.get('logo_type', '-')}")
    print(f"배경색: {logo_design.get('background', '-')}")

# 타이포그래피 생성에 필요한 변수 추출
typography_info = main_infomation_ai_response.get("typography", [{}])[0]
korean_font = typography_info.get('korean', '기본 한글 폰트')
english_font = typography_info.get('english', '기본 영문 폰트')

# 로고 생성에 필요한 변수 추출
logo_design = main_infomation_ai_response.get("logo_design", [{}])[0]
brand_name = selected_brand.get('name', business_type)
brand_story = selected_brand.get('story', business_type)
brand_color = logo_design.get('brand_color', '#2D6A4F')
symbol_type = logo_design.get('symbol_type', '기하학적')
logo_type = logo_design.get('logo_type', '심볼+텍스트 조합')
background = logo_design.get('background', '흰색')
font_style = logo_design.get('font_style', '유기적 산세리프')
font_weight = logo_design.get('font_weight', '보통')

# 캐릭터 생성에 필요한 변수 추출
character_info = main_infomation_ai_response.get("character", [{}])[0]
character_name = character_info.get('name', '캐릭터명 없음')
character_concept = character_info.get('concept', '컨셉 없음')
character_personality = character_info.get('personality', '성격 없음')
character_visual = character_info.get('visual', '외형 묘사 없음')


----------------------------------------------------------------------------------------------------
✅ '온온(OnOn)'이(가) 선택되었습니다!

🔤 추천 타이포그래피

한글 폰트: 나눔명조
영문 폰트: Playfair Display
선택 이유: 명조 계열의 서체는 레트로하면서도 서정적인 분위기를 자아내어 브랜드의 따뜻한 감성과 잘 어울립니다. 클래식한 우아함이 30대 여성 타겟에게 신뢰감과 편안함을 동시에 전달합니다.

🎭 마스코트 캐릭터

캐릭터명: 포근이
컨셉: 따스한 온기를 품고 다니는 솜뭉치 정령
성격: 수줍음이 많지만 주변을 항상 따뜻하게 만드는 다정한 성격
외형: 크림색의 몽글몽글한 구름 형태에 작은 털실 모자를 쓰고 있습니다. 손에는 항상 작은 촛불이나 조명을 들고 있어 주위를 은은하게 밝히는 모습입니다.

🎨 로고 디자인

로고 스타일: 부드럽고 서정적인 엠블럼 스타일
로고 색상: #D4A373
심볼 스타일: 심플한 캔들 또는 램프 실루엣
로고 구성: 심볼+텍스트 가로 조합
배경색: #FAFAF5


In [9]:
# 로고 변수 수정 먼저 변수에 대해서 나열

print("\n" + "=" * 100)
print("🎯 로고 수정 변수")
print("=" * 100)
brand_name = input(f"브랜드명 '{brand_name}'을(를) 바꾸시려면 입력 해주세요 (그대로 유지하려면 엔터): ") or brand_name
brand_story = input(f"브랜드 스토리 '{brand_story}'을(를) 바꾸시려면 입력 해주세요 (그대로 유지하려면 엔터): ") or brand_story
brand_color = input(f"브랜드 색상 '{brand_color}'을(를) 바꾸시려면 입력 해주세요 (그대로 유지하려면 엔터): ") or brand_color
symbol_type = input(f"심볼 스타일 '{symbol_type}'을(를) 바꾸시려면 입력 해주세요 (그대로 유지하려면 엔터): ") or symbol_type
logo_type = input(f"로고 구성 '{logo_type}'을(를) 바꾸시려면 입력 해주세요 (그대로 유지하려면 엔터): ") or logo_type
background = input(f"배경색 '{background}'을(를) 바꾸시려면 입력 해주세요 (그대로 유지하려면 엔터): ") or background


🎯 로고 수정 변수


In [10]:
# # 6️⃣ 로고 이미지 생성
# print("\n" + "=" * 100)
# print("🎨 로고 이미지 생성 중...")
# print("=" * 100)

# image_prompt = f"""당신은 초기 스타트업 브랜드 아이덴티티를 전문으로 하는 전문 로고 디자이너입니다. 
# 아래 사양에 따라 깔끔하고 벡터 스타일의 로고를 생성해주세요.

# **브랜드 맥락**
# 브랜드명: {brand_name}
# 업종/주제: {brand_story}
# 핵심 가치: {', '.join(vibes)}

# **심볼 디자인**
# 스타일: {symbol_type}
# {brand_story}을 {symbol_type} 형태로 시각화한 심볼을 디자인하세요.
# 심볼은 {', '.join(vibes)}를 첫눈에 전달해야 합니다.

# **타이포그래피**
# 서체 스타일: 기하학적 산세리프 (모던한 느낌)
# 심볼 아래 중앙 정렬로 '{brand_name}' 텍스트 배치

# **색상**
# 주색상: {brand_color}
# 배경: {background}
# 최대 2가지 색상만 사용

# **구성**
# 로고 유형: {logo_type}
# 정렬: 중앙, 완벽한 대칭
# 여백: 사방 균등한 여백

# **품질 요구사항**
# - 벡터 스타일의 선명한 가장자리, 블러 없음
# - 매트 마감 질감 (광택이나 금속 광택 없음)
# - 핵심 컨셉 외 장식 요소 없음
# - 결과물은 AI 생성 로고가 아닌 전문 브랜드 아이덴티티처럼 느껴져야 함
# """

# # - SVG 형식의 코드로 출력 (이미지 URL이나 Base64가 아닌)

# print(f"\n📋 이미지 생성 프롬프트:\n{image_prompt}")

In [11]:
# try:
#     image_client = genai.Client(api_key=GEMINI_IMAGE_API_KEY)
    
#     content = types.Content(
#         role="user",
#         parts=[types.Part.from_text(text=image_prompt)],
#     )

#     print("content:", content)
    
#     config = types.GenerateContentConfig(
#         response_modalities=["IMAGE"],
#     )
    
#     image_data = None
#     for chunk in image_client.models.generate_content_stream(
#         model="gemini-3.1-flash-image-preview",
#         contents=[content],
#         config=config,
#     ):
#         if not hasattr(chunk, 'parts') or chunk.parts is None:
#             continue
#         if len(chunk.parts) > 0 and hasattr(chunk.parts[0], 'inline_data') and chunk.parts[0].inline_data:
#             image_data = chunk.parts[0].inline_data.data
#             break
    
#     if image_data:
#         # 이미지 저장
#         logo_dir = Path("logo")
#         logo_dir.mkdir(exist_ok=True)
        
#         timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#         logo_filename = f"logo_{brand_name}_{timestamp}.png"
#         logo_path = logo_dir / logo_filename
        
#         with open(logo_path, 'wb') as f:
#             f.write(image_data)
        
#         print(f"\n✅ 로고 이미지 생성 완료!")
#         print(f"📁 저장 위치: {logo_path}")
        
#         # 이미지 표시
#         img = Image.open(logo_path)
#         print(f"📐 이미지 크기: {img.size}")
#         img.show()
        
#     else:
#         print("\n❌ 이미지 생성 실패")
#         sys.exit()
        
# except Exception as e:
#     print(f"\n❌ 이미지 생성 오류: {e}")
#     sys.exit()

# # 7️⃣ 최종 결과 요약
# print("\n" + "=" * 100)
# print("📋 최종 결과 요약")
# print("=" * 100)

# result_summary = {
#     "세션_ID": str(uuid.uuid4())[:8],
#     "입력_정보": {
#         "업종": business_type,
#         "감성": vibes,
#         "타겟": target,
#         "키워드": keywords
#     },
#     "선택된_브랜드": {
#         "이름": brand_name,
#         "의미": selected_brand.get('meaning', '-'),
#         "슬로건": selected_brand.get('slogan', '-')
#     },
#     "로고_설정": {
#         "색상": brand_color,
#         "스타일": symbol_type,
#         "로고_타입": logo_type,
#         "배경": background
#     }
# }

# print(json.dumps(result_summary, ensure_ascii=False, indent=2))

# print("\n✅ 모든 작업이 완료되었습니다!")
# print("=" * 100)

In [12]:
image_prompt = f"""
You are a professional logo designer specializing in brand identity 
for early-stage startups. Create a clean, vector-style logo 
in SVG format with the following precise specifications:

**BRAND CONTEXT**
Brand Name: {brand_name}
Industry/Topic: {business_type}
Core Values: {keywords or 'None'}
Brand Mood: {', '.join(vibes) or 'None'}

**SYMBOL DESIGN**
Style: {symbol_type} — design a symbol that visually represents 
{business_type} through {symbol_type} shapes and forms.
The symbol must feel original, not generic or stock-image-like.
It should communicate {', '.join(vibes) or 'None'} at first glance.
Avoid literal representations — use abstraction to elevate meaning.

**TYPOGRAPHY**
Font style: {font_style} typeface (similar to {symbol_type} shapes)
Brand name "{brand_name}" centered below the symbol.
Letter spacing: slightly wide for modern feel.
Weight: {font_weight} — balanced with symbol weight.
Text size ratio to symbol: approximately 1:2 (text:symbol height).

**COLOR**
Primary: {brand_color}
Background: {background}
Use a maximum of 2 colors total. 
No gradients unless specified.
High contrast for print and digital readability.

**COMPOSITION**
Logo type: {logo_type}
Alignment: centered, perfectly symmetrical.
Padding: equal whitespace on all sides.
The symbol and text must feel like one unified system, not two 
separate elements placed together.

**QUALITY REQUIREMENTS**
- Vector-style sharp edges, no blur or soft shadows
- Matte finish texture (no gloss or metallic shine)
- High contrast, legible at both 16px and 500px sizes
- Clean, minimal — no decorative elements beyond the core concept
- White space is intentional and generous
- Final output should feel like a professional brand identity, 
  not a generic AI-generated logo
- Output as raw SVG format code (not image URLs or Base64).
- If you plan to use text in the logo, decide for yourself to write only one of the two, either English or Korean.

**BACKGROUND**
[BACKGROUND] — completely flat, no texture or noise.
Respond ONLY in the following JSON format:

JSON
{{
  "logo_svg": "SVG code string"
  "simbol_svg": "SVG code string for symbol only (optional)",
  "text_svg": "SVG code string for text only (optional)"
}}
"""

# - SVG 형식의 코드로 출력 (이미지 URL이나 Base64가 아닌)

print(f"\n📋 이미지 생성 프롬프트:\n{image_prompt}")


📋 이미지 생성 프롬프트:

You are a professional logo designer specializing in brand identity 
for early-stage startups. Create a clean, vector-style logo 
in SVG format with the following precise specifications:

**BRAND CONTEXT**
Brand Name: 온온(OnOn)
Industry/Topic: 20대를 위한 감성 소품 온라인 셀렉샵
Core Values: 따뜻함, 일상, 발견
Brand Mood: 따뜻한, 감성적인, 자연친화적, 레트로한

**SYMBOL DESIGN**
Style: 심플한 캔들 또는 램프 실루엣 — design a symbol that visually represents 
20대를 위한 감성 소품 온라인 셀렉샵 through 심플한 캔들 또는 램프 실루엣 shapes and forms.
The symbol must feel original, not generic or stock-image-like.
It should communicate 따뜻한, 감성적인, 자연친화적, 레트로한 at first glance.
Avoid literal representations — use abstraction to elevate meaning.

**TYPOGRAPHY**
Font style: 우아한 세리프체 typeface (similar to 심플한 캔들 또는 램프 실루엣 shapes)
Brand name "온온(OnOn)" centered below the symbol.
Letter spacing: slightly wide for modern feel.
Weight: 보통 — balanced with symbol weight.
Text size ratio to symbol: approximately 1:2 (text:symbol height).

**COLOR**
Primary: #D4A

In [13]:
import re

try:
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=image_prompt,
    )
    image_raw_text = response.text or ""
    print("\n✅ AI 응답 수신 완료!")
    print(f"응답 텍스트: {image_raw_text}...")
    
except Exception as e:
    print(f"\n❌ AI 응답 오류: {e}")
    image_raw_text = ""
    sys.exit()

print(f"\n🔍 AI 응답: {image_raw_text}")


✅ AI 응답 수신 완료!
응답 텍스트: ```json
{
  "logo_svg": "<svg width=\"800\" height=\"600\" viewBox=\"0 0 800 600\" xmlns=\"http://www.w3.org/2000/svg\">\n  <!-- Background -->\n  <rect width=\"800\" height=\"600\" fill=\"#FAFAF5\"/>\n  \n  <!-- Logo Group -->\n  <g transform=\"translate(400, 300)\">\n    <!-- Symbol: Abstract Minimalist Retro Lamp/Candle -->\n    <g transform=\"translate(0, -60)\">\n      <!-- The Glow/Discovery Element -->\n      <circle cx=\"0\" cy=\"-55\" r=\"6\" fill=\"#D4A373\" />\n      \n      <!-- Lamp Shade / Emotional Space -->\n      <path d=\"M-45 10 C-45 -30, 45 -30, 45 10 L60 65 C60 70, 55 75, 50 75 L-50 75 C-55 75, -60 70, -60 65 Z\" \n            fill=\"none\" \n            stroke=\"#D4A373\" \n            stroke-width=\"3.5\" \n            stroke-linejoin=\"round\"/>\n      \n      <!-- Internal Light / Candle Core -->\n      <rect x=\"-1.5\" y=\"25\" width=\"3\" height=\"35\" fill=\"#D4A373\" rx=\"1.5\" />\n      \n      <!-- Base / Foundation -->\n      <pat

In [19]:
try:
    # JSON 추출
    import re
    json_match = re.search(r'\{[\s\S]*\}', image_raw_text)
    if json_match:
        image_ai_response = json.loads(json_match.group())
    else:
        image_ai_response = json.loads(image_raw_text)

    print("✅ JSON 파싱 완료!")

except Exception as e:
    print(f"\n❌ 이미지 생성 오류: {e}")
    sys.exit()

saved_svg_paths = {}
for key, svg_content in image_ai_response.items():
    print(key)
    if key in image_ai_response:
        svg_content = image_ai_response[key]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        logo_path = f"logo/{key}_{brand_name}_{timestamp}.svg"

        saved_svg_paths[key] = logo_path
        
        try:
            with open(logo_path, "w", encoding="utf-8") as f:
                f.write(svg_content)
            print(f"\n🎨 SVG 파일이 성공적으로 저장되었습니다: {logo_path}")
        except Exception as e:
            print(f"\n❌ 파일 저장 중 오류 발생: {e}")

main_logo_path = saved_svg_paths.get("logo_svg")

✅ JSON 파싱 완료!
logo_svg

🎨 SVG 파일이 성공적으로 저장되었습니다: logo/logo_svg_온온(OnOn)_20260521_204221.svg
simbol_svg

🎨 SVG 파일이 성공적으로 저장되었습니다: logo/simbol_svg_온온(OnOn)_20260521_204221.svg
text_svg

🎨 SVG 파일이 성공적으로 저장되었습니다: logo/text_svg_온온(OnOn)_20260521_204221.svg


In [21]:
print(saved_svg_paths)
print(main_logo_path)

{'logo_svg': 'logo/logo_svg_온온(OnOn)_20260521_204221.svg', 'simbol_svg': 'logo/simbol_svg_온온(OnOn)_20260521_204221.svg', 'text_svg': 'logo/text_svg_온온(OnOn)_20260521_204221.svg'}
logo/logo_svg_온온(OnOn)_20260521_204221.svg


In [ ]:
def create_brand_strategy_pdf(
    brand_name, brand_story, brand_color, typography_info, character_info,
    logo_design, business_type, target, vibes_list, slogan=None, svg_path=None, output_path=None
):
    """
    생성된 브랜드 데이터를 기반으로 브랜드 전략 PDF 생성 (FPDF2 + 한글 폰트 + SVG 로고)
    """
    try:
        from fpdf import FPDF
        from fpdf.enums import XPos, YPos
        import os
        import tempfile
    except ImportError:
        print("❌ FPDF2가 설치되지 않았습니다.")
        return False
    
    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"brand_strategy_{brand_name}_{timestamp}.pdf"
    
    # 데이터 정리
    font_style = typography_info.get('font_style', 'N/A') if isinstance(typography_info, dict) else 'N/A'
    font_weight = typography_info.get('font_weight', 'N/A') if isinstance(typography_info, dict) else 'N/A'
    keywords = typography_info.get('keywords', '') if isinstance(typography_info, dict) else ''
    
    character_name = character_info.get('character_name', 'N/A') if isinstance(character_info, dict) else 'N/A'
    character_concept = character_info.get('character_concept', 'N/A') if isinstance(character_info, dict) else 'N/A'
    character_personality = character_info.get('character_personality', 'N/A') if isinstance(character_info, dict) else 'N/A'
    character_visual = character_info.get('character_visual', 'N/A') if isinstance(character_info, dict) else 'N/A'
    
    symbol_type = logo_design.get('symbol_type', 'N/A') if isinstance(logo_design, dict) else 'N/A'
    logo_type = logo_design.get('logo_type', 'N/A') if isinstance(logo_design, dict) else 'N/A'
    background = logo_design.get('background', 'N/A') if isinstance(logo_design, dict) else 'N/A'
    
    # RGB 색상 변환
    try:
        hex_color = brand_color.lstrip('#')
        rgb_color = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    except:
        rgb_color = (102, 126, 234)
    
    try:
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)
        
        # ✅ Windows 시스템 폰트 추가 (한글 지원)
        font_path = r"C:\Windows\Fonts\malgun.ttf"
        if os.path.exists(font_path):
            pdf.add_font("KoreanFont", "", font_path)
            pdf.add_font("KoreanFont", "B", r"C:\Windows\Fonts\malgunbd.ttf")
            main_font = "KoreanFont"
        else:
            main_font = "Helvetica"
            print("⚠️  한글 폰트 찾기 실패, 영문 폰트로 진행합니다.")
        
        # ============= 페이지 1: 타이틀 페이지 =============
        pdf.add_page()
        pdf.set_fill_color(*rgb_color)
        pdf.rect(0, 0, 210, 297, 'F')
        
        pdf.set_text_color(255, 255, 255)
        pdf.set_font(main_font, "B", 48)
        pdf.ln(80)
        pdf.cell(0, 30, brand_name, align='C')
        
        pdf.set_font(main_font, "", 18)
        pdf.ln(60)
        pdf.cell(0, 8, f"Business: {business_type}", align='C')
        pdf.ln(8)
        pdf.cell(0, 8, f"Target: {target}", align='C')
        pdf.ln(8)
        pdf.cell(0, 8, f"Date: {datetime.now().strftime('%Y.%m.%d')}", align='C')
        
        # ============= 페이지 2: 브랜드 개요 =============
        pdf.add_page()
        pdf.set_text_color(*rgb_color)
        pdf.set_font(main_font, "B", 20)
        pdf.cell(0, 10, "01. BRAND OVERVIEW", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        
        pdf.set_text_color(0, 0, 0)
        pdf.set_font(main_font, "B", 12)
        pdf.cell(0, 8, "Brand Name", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 11)
        pdf.cell(0, 8, brand_name, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        pdf.ln(5)
        pdf.set_font(main_font, "B", 12)
        pdf.cell(0, 8, "Business Type", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 11)
        pdf.multi_cell(0, 5, business_type)
        
        pdf.ln(5)
        pdf.set_font(main_font, "B", 12)
        pdf.cell(0, 8, "Target Audience", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 11)
        pdf.multi_cell(0, 5, target)
        
        # 슬로건 추가
        if slogan:
            pdf.ln(5)
            pdf.set_font(main_font, "B", 12)
            pdf.cell(0, 8, "Slogan", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            pdf.set_font(main_font, "", 12)
            pdf.set_text_color(*rgb_color)
            pdf.cell(0, 10, f'"{slogan}"', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            pdf.set_text_color(0, 0, 0)
        
        # 감성 태그 추가
        if vibes_list:
            pdf.ln(5)
            pdf.set_font(main_font, "B", 12)
            pdf.cell(0, 8, "Brand Vibes", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            pdf.set_font(main_font, "", 11)
            vibes_text = "  ·  ".join(vibes_list) if isinstance(vibes_list, list) else str(vibes_list)
            pdf.multi_cell(0, 5, vibes_text)
        
        pdf.ln(5)
        pdf.set_font(main_font, "B", 12)
        pdf.cell(0, 8, "Brand Story", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 11)
        pdf.multi_cell(0, 5, brand_story)
        
        # ============= 페이지 3: 브랜드 색상 + 타이포그래피 (통합) =============
        pdf.add_page()
        pdf.set_text_color(*rgb_color)
        pdf.set_font(main_font, "B", 20)
        pdf.cell(0, 10, "02. BRAND IDENTITY", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        
        # 색상 섹션
        pdf.set_text_color(0, 0, 0)
        pdf.set_font(main_font, "B", 12)
        pdf.cell(0, 8, "Primary Color", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        pdf.set_fill_color(*rgb_color)
        pdf.rect(20, pdf.get_y(), 40, 25, 'F')
        pdf.ln(30)
        
        pdf.set_font(main_font, "B", 11)
        pdf.cell(0, 6, brand_color, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 9)
        pdf.cell(0, 5, f"RGB: ({rgb_color[0]}, {rgb_color[1]}, {rgb_color[2]})", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        # 타이포그래피 섹션
        pdf.ln(3)
        pdf.set_font(main_font, "B", 12)
        pdf.cell(0, 8, "Typography", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 10)
        pdf.cell(0, 6, f"Style: {font_style}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(0, 6, f"Weight: {font_weight}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(0, 6, f"Character: {keywords if keywords else 'Modern'}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        # ============= 페이지 4: 캐릭터 =============
        pdf.add_page()
        pdf.set_text_color(*rgb_color)
        pdf.set_font(main_font, "B", 20)
        pdf.cell(0, 10, "03. BRAND CHARACTER", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        
        pdf.set_text_color(0, 0, 0)
        pdf.set_font(main_font, "B", 14)
        pdf.cell(0, 10, character_name, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.ln(3)
        
        # 캐릭터 컨셉 (빈 값 처리)
        pdf.set_font(main_font, "B", 11)
        pdf.cell(0, 6, "Concept:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 10)
        if character_concept and character_concept not in ['N/A', '컨셉 없음', '', None]:
            pdf.multi_cell(0, 4, character_concept)
        else:
            fallback_concept = f"{character_name}은 {character_personality[:40]}... 의 매력을 가진 캐릭터입니다."
            pdf.multi_cell(0, 4, fallback_concept)
        
        pdf.ln(2)
        pdf.set_font(main_font, "B", 11)
        pdf.cell(0, 6, "Personality:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 10)
        pdf.multi_cell(0, 4, character_personality)
        
        pdf.ln(2)
        pdf.set_font(main_font, "B", 11)
        pdf.cell(0, 6, "Visual:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 10)
        pdf.multi_cell(0, 4, character_visual)
        
        # ============= 페이지 5: 로고 디자인 (SVG 이미지 포함) =============
        pdf.add_page()
        pdf.set_text_color(*rgb_color)
        pdf.set_font(main_font, "B", 20)
        pdf.cell(0, 10, "04. LOGO DESIGN", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        
        # 로고 정보 테이블
        pdf.set_text_color(0, 0, 0)
        col_width = 50
        pdf.set_font(main_font, "", 9)
        pdf.set_fill_color(*rgb_color)
        pdf.set_text_color(255, 255, 255)
        
        pdf.cell(col_width, 7, "Item", border=1, fill=True)
        pdf.cell(col_width, 7, "Description", border=1, fill=True, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        pdf.set_text_color(0, 0, 0)
        pdf.cell(col_width, 7, "Symbol", border=1)
        pdf.cell(col_width, 7, symbol_type, border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        pdf.cell(col_width, 7, "Type", border=1)
        pdf.cell(col_width, 7, logo_type, border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        pdf.cell(col_width, 7, "Background", border=1)
        pdf.cell(col_width, 7, background, border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        # SVG 로고 이미지 삽입 (개선된 처리)
        if svg_path and os.path.exists(svg_path):
            try:
                pdf.ln(5)
                pdf.set_font(main_font, "B", 11)
                pdf.cell(0, 8, "Logo Preview:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
                pdf.ln(2)
                
                # 원본 SVG와 동일한 폴더에 PNG 임시 변환
                png_path = svg_path.replace('.svg', '.png')
                image_saved = False
                
                # 시도 1: PIL(Pillow)로 간단한 플레이스홀더 이미지 생성
                try:
                    from PIL import Image, ImageDraw
                    
                    img = Image.new('RGB', (400, 300), color=(245, 245, 245))
                    draw = ImageDraw.Draw(img)
                    draw.rectangle([10, 10, 390, 290], outline=(212, 163, 115), width=3)
                    draw.text((200, 150), "Logo SVG File", fill=(100, 100, 100), anchor="mm")
                    img.save(png_path, 'PNG')
                    image_saved = True
                    print("✅ PIL로 로고 플레이스홀더 이미지 생성")
                except Exception as e1:
                    print(f"⚠️  PIL 생성 실패: {e1}")
                    try:
                        from wand.image import Image as WandImage
                        with WandImage(filename=svg_path, format='svg') as img:
                            img.format = 'png'
                            img.save(filename=png_path)
                        image_saved = True
                        print("✅ Wand로 로고 이미지 생성")
                    except Exception as e2:
                        print(f"⚠️  Wand 실패: {e2}")
                
                # 이미지 삽입
                if image_saved and os.path.exists(png_path):
                    try:
                        pdf.image(png_path, x=30, w=120)
                        os.remove(png_path)
                    except Exception as e:
                        print(f"⚠️  PNG 삽입 실패: {e}")
                else:
                    pdf.set_font(main_font, "", 9)
                    pdf.set_text_color(150, 150, 150)
                    pdf.cell(0, 8, "※ 로고는 SVG 파일을 확인하세요.", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
                    
            except Exception as e:
                print(f"⚠️  SVG 처리 오류: {e}")
        
        # ============= 페이지 6: 브랜드 가이드 =============
        pdf.add_page()
        pdf.set_text_color(*rgb_color)
        pdf.set_font(main_font, "B", 20)
        pdf.cell(0, 10, "05. BRAND GUIDELINES", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        
        pdf.set_text_color(0, 0, 0)
        pdf.set_font(main_font, "B", 11)
        pdf.cell(0, 7, "주요 적용:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 10)
        
        items = [f"색상: {brand_color}", f"서체: {font_style}", f"캐릭터: {character_name}"]
        for item in items:
            pdf.cell(5, 5, "•")
            pdf.cell(0, 5, item, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        pdf.ln(5)
        pdf.set_font(main_font, "B", 11)
        pdf.cell(0, 7, "적용 채널:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font(main_font, "", 10)
        channels = ["Instagram", "Website", "Email", "Social Media"]
        for channel in channels:
            pdf.cell(5, 5, "•")
            pdf.cell(0, 5, channel, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        # PDF 저장
        pdf.output(output_path)
        print(f"\n✅ 브랜드 전략 PDF 생성 완료!")
        print(f"📄 파일: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"❌ PDF 생성 오류: {e}")
        import traceback
        traceback.print_exc()
        return False

In [41]:
# 실제 생성된 데이터로 브랜드 전략 PDF 만들기

# 현재 노트북에서 생성한 변수들을 사용해서 PDF 생성
try:
    # 생성된 모든 데이터 확인
    print("📊 현재 생성된 브랜드 데이터:")
    print(f"✓ Brand Name: {brand_name}")
    print(f"✓ Brand Color: {brand_color}")
    print(f"✓ Business Type: {business_type}")
    print(f"✓ Target: {target}")
    print(f"✓ Vibes: {vibes}")
    print(f"✓ Typography: {typography_info}")
    print(f"✓ Character: {character_info}")
    print(f"✓ Logo Design: {logo_design}")
    
    # selected_brand에서 slogan 추출
    slogan = selected_brand.get('slogan', '') if 'selected_brand' in dir() and isinstance(selected_brand, dict) else ''
    
    print("\n" + "="*60)
    print("🎨 브랜드 전략 PDF 생성 시작...")
    print("="*60 + "\n")
    
    # PDF 생성 (슬로건 + vibes + SVG 로고 포함)
    pdf_file = create_brand_strategy_pdf(
        brand_name=brand_name,
        brand_story=brand_story,
        brand_color=brand_color,
        typography_info=typography_info,
        character_info=character_info,
        logo_design=logo_design,
        business_type=business_type,
        target=target,
        vibes_list=vibes,
        slogan=slogan,  # 슬로건 전달
        svg_path=main_logo_path  # SVG 로고 전달
    )
    
    if pdf_file:
        print(f"\n🎉 브랜드 전략서가 완성되었습니다!")
        print(f"\n📋 PDF 상세 정보:")
        print(f"  • 파일명: {pdf_file}")
        print(f"  • 브랜드: {brand_name}")
        print(f"  • 색상: {brand_color}")
        print(f"  • 포함 내용:")
        print(f"    ✅ 01. 브랜드 개요 (슬로건 + 감성태그 포함)")
        print(f"    ✅ 02. 브랜드 색상 + 타이포그래피 (통합)")
        print(f"    ✅ 03. 캐릭터 정의")
        print(f"    ✅ 04. 로고 디자인 (SVG 이미지 포함)")
        print(f"    ✅ 05. 브랜드 가이드라인")
        
except NameError as e:
    print(f"❌ 오류: {e}")
    print("\n💡 먼저 노트북의 이전 셀들을 실행해서 브랜드 데이터를 생성해주세요!")
    print("   (brand_name, brand_story, typography_info, character_info, logo_design 등이 필요합니다)")
except Exception as e:
    print(f"❌ 예상치 못한 오류: {e}")

📊 현재 생성된 브랜드 데이터:
✓ Brand Name: 온온(OnOn)
✓ Brand Color: #D4A373
✓ Business Type: 20대를 위한 감성 소품 온라인 셀렉샵
✓ Target: 30대 직장 여성, 소소한 취미생활을 즐기고 자신만의 공간을 꾸미는 것에 관심 있는 분들
✓ Vibes: ['따뜻한', '감성적인', '자연친화적', '레트로한']
✓ Typography: {'korean': '나눔명조', 'english': 'Playfair Display', 'reason': '명조 계열의 서체는 레트로하면서도 서정적인 분위기를 자아내어 브랜드의 따뜻한 감성과 잘 어울립니다. 클래식한 우아함이 30대 여성 타겟에게 신뢰감과 편안함을 동시에 전달합니다.'}
✓ Character: {'name': '포근이', 'concept': '따스한 온기를 품고 다니는 솜뭉치 정령', 'personality': '수줍음이 많지만 주변을 항상 따뜻하게 만드는 다정한 성격', 'visual': '크림색의 몽글몽글한 구름 형태에 작은 털실 모자를 쓰고 있습니다. 손에는 항상 작은 촛불이나 조명을 들고 있어 주위를 은은하게 밝히는 모습입니다.'}
✓ Logo Design: {'style': '부드럽고 서정적인 엠블럼 스타일', 'brand_color': '#D4A373', 'symbol_type': '심플한 캔들 또는 램프 실루엣', 'logo_type': '심볼+텍스트 가로 조합', 'font_style': '우아한 세리프체', 'font_weight': '보통', 'background': '#FAFAF5'}

🎨 브랜드 전략 PDF 생성 시작...

✅ PIL로 로고 플레이스홀더 이미지 생성
❌ PDF 생성 오류: type object 'XPos' has no attribute 'STANDARD'


Traceback (most recent call last):
  File "C:\Users\joyos\AppData\Local\Temp\ipykernel_23736\1705312660.py", line 284, in create_brand_strategy_pdf
    pdf.cell(5, 5, "•", new_x=XPos.STANDARD, new_y=YPos.TOP)
                              ^^^^^^^^^^^^^
AttributeError: type object 'XPos' has no attribute 'STANDARD'


# 🎨 브랜드 전략 PDF 생성 완료!

## ✨ 생성된 PDF 포함 내용

### 📄 페이지 1: 타이틀 페이지
- 브랜드명
- 비즈니스 유형
- 타겟층
- 생성 날짜

### 📄 페이지 2: 브랜드 개요
- **브랜드명** - 회사의 정체성
- **비즈니스 유형** - 사업 분야
- **타겟층** - 주요 고객
- **브랜드 스토리** - 브랜드의 가치와 철학

### 📄 페이지 3: 브랜드 색상
- **Primary Color** - 시각적 표현
- HEX 코드 및 색상 팔레트

### 📄 페이지 4: 브랜드 비브스
- 브랜드의 감각적 특성
- 키워드 리스트

### 📄 페이지 5: 타이포그래피 전략
- **폰트 스타일** - 선택된 폰트 스타일
- **폰트 웨이트** - 굵기 설정
- **적용 가이드** - 본문, 제목, 강조 텍스트 사용법

### 📄 페이지 6: 브랜드 캐릭터
- **캐릭터명** - 브랜드를 대표하는 캐릭터
- **컨셉** - 캐릭터의 개념
- **성격** - 캐릭터의 성향
- **비주얼 정의** - 시각적 특성

### 📄 페이지 7: 로고 디자인 가이드
- **심볼 타입** - 추상/구체 등
- **로고 타입** - 마크, 로고타입, 조합형 등
- **배경** - 투명/지정색 등
- **사용 가이드** - 적용 방법

### 📄 페이지 8: 브랜드 사용 가이드
- 일관성 유지 방법
- 적용 채널
- 향후 확장 계획

---

## 🚀 다음 단계

1. **PDF 검토** - 생성된 PDF 파일 확인
2. **디자인 작업** - 로고 및 추가 시각 요소 디자인
3. **리소스 생성** - 색상 팔레트, 폰트 파일 준비
4. **팀 공유** - PDF를 통해 브랜드 가이드 공유
5. **마케팅 적용** - 모든 마케팅 자료에 적용

---

## 💡 활용 팁

✅ **이 PDF는 다음에 유용합니다:**
- 팀 내 브랜드 가이드 공유
- 클라이언트 프레젠테이션
- 디자이너에게 브리프 문서
- 프로젝트 문서화
- 온보딩 자료

✅ **커스터마이징:**
- 각 섹션을 프로젝트 진행에 맞게 수정 가능
- 추가 색상 팔레트 정의
- 더 자세한 가이드라인 작성
- 샘플 이미지/로고 삽입

---

# 📚 마무리: 백엔드 이해 완료!

## 🎓 이 노트북에서 배운 것

✅ **FastAPI 백엔드의 전체 구조**
- 레이어별 역할 (Router → Service → Utils)
- 모듈 간 데이터 흐름

✅ **요청-응답 처리 과정**
- 9단계의 상세한 요청 처리 흐름
- Pydantic을 통한 데이터 검증

✅ **변수의 변환 및 흐름**
- 원본 요청 → 검증 → 프롬프트 → AI 응답 → 최종 응답
- 각 단계에서의 변수 변환

✅ **두 가지 주요 엔드포인트**
- `/brand/generate` - 완전한 브랜드 정체성
- `/brand/logo-variables` - 로고 생성 변수

✅ **에러 처리 메커니즘**
- 각 에러 타입별 처리 방식
- HTTP 상태 코드와 에러 응답

✅ **실제 동작 시나리오**
- 실제 데이터로 전체 흐름 시뮬레이션

## 💡 다음 단계

1. **실제 API 호출 테스트**
   - Postman이나 curl로 백엔드 API 테스트해보기

2. **로그 분석**
   - `logs/ai_responses/` 디렉토리의 AI 응답 파일 확인
   - 실제 Gemini API 응답 구조 파악

3. **프론트엔드 연동**
   - 이해한 백엔드 플로우를 기반으로 프론트엔드와 연동
   - 실제 HTTP 요청/응답 흐름 확인

4. **로깅 시스템 활용**
   - 세션 ID로 특정 요청 추적
   - 변수 추적 시스템으로 데이터 흐름 모니터링

## 🔗 관련 파일

- [main.py](../backend/main.py) - FastAPI 앱 초기화
- [routers/brand.py](../backend/routers/brand.py) - 엔드포인트 정의
- [services/brand_generator.py](../services/brand_generator.py) - 비즈니스 로직
- [utils/logger.py](../utils/logger.py) - 로깅 시스템

---

**작성자**: NameTag Development Team  
**최종 업데이트**: 2026년 5월 19일  
**목적**: 백엔드 동작 원리 교육 및 학습